# 01 DB Population

Purpose: Populate persistent ChromaDB image/text collections from training data (idempotent).

Inputs: `dataset/CVPR_2024_dataset_Train/`, `dataset_text/train.csv`.

Outputs: Persistent collections in `chroma_db/` (`trash_image_db`, `trash_text_db`).

In [3]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../..").resolve()))

import numpy as np
from PIL import Image
from tqdm.auto import tqdm

from src.phase2.config import get_phase2_config
from src.phase2.data_utils import build_records_from_csv
from src.phase2.db_client import (
    get_image_collection,
    get_persistent_client,
    get_text_collection,
)

d:\University of Calgary\Winter 2026\ENSF617-Introduction-To-Machine-Learning\trash-classification-project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
CONFIG = get_phase2_config()

REPO_ROOT = Path("../..").resolve()
TRAIN_DIR = REPO_ROOT / "dataset" / "CVPR_2024_dataset_Train"
TRAIN_CSV = REPO_ROOT / "dataset_text" / "train.csv"

if not TRAIN_DIR.exists() or not TRAIN_CSV.exists():
    raise FileNotFoundError(
        "Training dataset missing. Expected dataset and dataset_text paths in repo root."
    )

records, missing_examples, total_rows = build_records_from_csv(
    csv_path=TRAIN_CSV,
    split_dir=TRAIN_DIR,
    text_column="text",
    label_column="label",
    text_key="filename",
)

if missing_examples:
    print("Skipped rows with missing image files (up to 10 shown):")
    for item in missing_examples:
        print(f"  - {item}")

print(f"Training records available for DB population: {len(records)} / {total_rows}")

d:\University of Calgary\Winter 2026\ENSF617-Introduction-To-Machine-Learning\trash-classification-project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Training records available for DB population: 11685 / 11685


In [ ]:
try:
    client = get_persistent_client(str(REPO_ROOT / "chroma_db"))
    image_collection = get_image_collection(client)
    text_collection = get_text_collection(client)
except Exception as exc:
    raise RuntimeError(f"ChromaDB initialization failed: {exc}") from exc

expected_count = len(records)
if (
    image_collection.count() == expected_count
    and text_collection.count() == expected_count
):
    print("Collections already populated. Skipping insertion.")
    SKIP_POPULATION = True
else:
    SKIP_POPULATION = False

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 663.55it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
if not SKIP_POPULATION:
    batch_size = CONFIG["batch_size"]
    for start in tqdm(range(0, len(records), batch_size), desc="Populating ChromaDB"):
        batch = records[start : start + batch_size]
        ids_img, ids_txt = [], []
        images, docs = [], []
        metadatas = []

        for idx, record in enumerate(batch, start=start):
            image = Image.open(record["image_path"]).convert("RGB")
            image_np = np.array(image, dtype=np.uint8)
            ids_img.append(f"img_{idx}")
            ids_txt.append(f"txt_{idx}")
            images.append(image_np)
            docs.append(record["filename"])
            metadatas.append({"label": record["label"], "filename": record["filename"]})

        try:
            image_collection.add(ids=ids_img, images=images, metadatas=metadatas)
            text_collection.add(ids=ids_txt, documents=docs, metadatas=metadatas)
        except Exception as exc:
            raise RuntimeError(
                f"ChromaDB insert failed for batch starting at {start}: {exc}"
            ) from exc

print("Population step complete.")

Populating ChromaDB: 100%|██████████| 117/117 [24:53<00:00, 12.77s/it]

Population step complete.


In [4]:
print(f"Image collection count: {image_collection.count()}")
print(f"Text collection count: {text_collection.count()}")

Image collection count: 11685
Text collection count: 11685


In [ ]:
# Get mean, std, min, max of pairwise cosine distances within each class

import numpy as np

REPO_ROOT = Path("../..").resolve()
TRAIN_DIR = REPO_ROOT / "dataset" / "CVPR_2024_dataset_Train"
TRAIN_CSV = REPO_ROOT / "dataset_text" / "train.csv"

try:
    client = get_persistent_client(str(REPO_ROOT / "chroma_db"))
    image_collection = get_image_collection(client)
    text_collection = get_text_collection(client)
except Exception as exc:
    raise RuntimeError(f"ChromaDB initialization failed: {exc}") from exc

# After populating your ChromaDB, fetch all embeddings per class
# Then compute pairwise cosine distances within each class

for class_name in ["Black", "Blue", "Green", "TTR"]:
    results = image_collection.get(where={"label": class_name}, include=["embeddings"])
    embeddings = np.array(results["embeddings"])

    # Pairwise cosine distances (sample if too large)
    sample = embeddings[:200]  # keep it tractable
    norms = np.linalg.norm(sample, axis=1, keepdims=True)
    normalized = sample / norms
    sim_matrix = normalized @ normalized.T
    dist_matrix = 1 - sim_matrix  # cosine distance

    # Upper triangle only (no self-pairs)
    upper = dist_matrix[np.triu_indices(len(sample), k=1)]
    print(
        f"{class_name}: mean={upper.mean():.3f}, "
        f"std={upper.std():.3f}, "
        f"min={upper.min():.3f}, "
        f"max={upper.max():.3f}"
    )

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 856.79it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Black: mean=0.556, std=0.117, min=0.075, max=0.945
Blue: mean=0.526, std=0.110, min=0.100, max=0.940
Green: mean=0.459, std=0.104, min=0.086, max=0.848
TTR: mean=0.613, std=0.127, min=0.095, max=0.970
